In [1]:
import sys
sys.path.append("src")

In [2]:
import os
import json
import string

import chromadb

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

from rag.vector_db import ChromaDB
from rank_bm25 import BM25Okapi

# Test DB

In [3]:
client = chromadb.HttpClient(host="localhost", port=8008)
collection = client.get_or_create_collection(name="rag-collection")

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
INFO:httpx:HTTP Request: GET http://localhost:8008/api/v2/auth/identity "HTTP/1.1 200 OK"
INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
INFO:httpx:HTTP Request: GET http://localhost:8008/api/v2/tenants/default_tenant "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8008/api/v2/tenants/default_tenant/databases/default_database "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8008/api/v2/tenants/default_tenant/databases/default_database/collections "HTTP/1.1 200 OK"


# Data

In [4]:
# sample_txt_data = "data/news_articles/05-03-ai-powered-supply-chain-startup-pando-lands-30m-investment.txt"

# with open(sample_txt_data, "r") as f:
#     file = f.read()

In [5]:
mami_bubu = "data/mami_bubu/"

data = []

for i, filename in enumerate(os.listdir(mami_bubu)):
    file_path = os.path.join(mami_bubu, filename)
    with open(file_path, "r") as f:
        file = json.load(f)

    data.append({
        "id": str(i),
        "filename": filename,
        "text": str(file)
    })

In [6]:
type(data[0]["text"])

str

In [7]:
for d in data:
    print(len(d["text"]))

1814
552
542
2087
1862
2933
2989
3825
1844
1796
2398
3546
805
792
1809
552
818
6422
5801
1820
3275
1788
1775
2297
542
2154
4085
2399
590
7317
3013
3292
1854
2440
542
1796
5359
1394
1775
7167
3213
682
3212
1796
1796
1829
552
1395
4502
2257
2257
436
3003


# vectorizer

In [8]:
# model = SentenceTransformer(
#     "mixedbread-ai/mxbai-embed-large-v1",
#     truncate_dim=512
# )
# model.save_pretrained("weights/mxbai-embeddings/")

In [9]:
model = SentenceTransformer("weights/mxbai-embeddings/", truncate_dim=512)

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: weights/mxbai-embeddings/
INFO:sentence_transformers.SentenceTransformer:2 prompts are loaded, with the keys: ['query', 'passage']


In [10]:
model.encode(data[0]["text"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

array([ 3.41595262e-02,  6.69061020e-02, -1.36620626e-01, -9.86907184e-02,
        2.48360962e-01, -9.66189563e-01,  9.40694869e-01,  3.22800577e-01,
        4.86894459e-01,  4.31055099e-01,  8.63419354e-01,  2.96319753e-01,
        3.88111770e-01,  6.15825877e-02, -1.39017999e-01,  6.77861094e-01,
        3.60838063e-02, -6.06617033e-01, -4.77675468e-01, -1.20995820e-01,
        1.56920508e-01,  3.36562365e-01, -1.99892223e+00, -7.24690795e-01,
       -1.50045991e-01,  1.01344216e+00, -5.02926260e-02, -6.48455977e-01,
        1.07830751e+00,  4.44274068e-01, -4.55600351e-01, -2.40814894e-01,
        3.04350853e-01, -3.14067394e-01,  4.76710737e-01,  6.27287149e-01,
        5.92369020e-01, -3.30930233e-01,  1.67620525e-01, -1.24574983e+00,
       -1.19477913e-01,  9.20927376e-02,  1.03741884e+00, -5.22228301e-01,
       -2.26259112e-01, -2.85838664e-01,  5.03813267e-01, -3.64802718e-01,
        6.00478947e-01, -5.49199581e-01, -7.09681511e-02, -3.05501610e-01,
        4.58750546e-01, -

# hybrid search

In [11]:
import re

In [12]:
db = ChromaDB(collection_name="rag-collection")

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
INFO:httpx:HTTP Request: GET http://localhost:8008/api/v2/auth/identity "HTTP/1.1 200 OK"
INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
INFO:httpx:HTTP Request: GET http://localhost:8008/api/v2/tenants/default_tenant "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8008/api/v2/tenants/default_tenant/databases/default_database "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8008/api/v2/tenants/default_tenant/databases/default_database/collections "HTTP/1.1 200 OK"


In [13]:
sample_query = "ara casual button set"

query = model.encode(sample_query)
query

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

array([-2.81063288e-01,  8.76076162e-01, -2.11922362e-01,  3.67036052e-02,
       -6.02897763e-01, -6.41735345e-02,  7.99056292e-02, -1.87151711e-02,
        1.10000111e-01,  3.37760955e-01,  3.42975318e-01,  5.61882913e-01,
       -5.09156168e-01, -7.53501952e-01,  1.05399899e-01, -2.10388020e-01,
       -6.60230756e-01, -6.29787683e-01, -3.60617757e-01,  4.72554803e-01,
        7.66631126e-01,  2.71660358e-01, -1.36910057e+00, -1.22302544e+00,
       -1.73907414e-01,  4.56063986e-01, -1.15260303e-01, -1.31431356e-01,
        1.88763648e-01,  6.92664504e-01, -1.89813331e-01,  1.86251417e-01,
        3.46603632e-01, -1.05229266e-01, -7.15195060e-01, -5.88669926e-02,
        4.44801003e-02, -4.79013532e-01, -7.96607494e-01, -6.09237254e-01,
        1.43204927e+00, -4.27263975e-02,  2.65476942e-01, -3.53142977e-01,
       -7.89960325e-01,  1.96688861e-01,  7.10347369e-02, -1.66071281e-01,
        5.53458214e-01, -7.54133999e-01, -7.15528652e-02,  1.47211134e-01,
       -3.47485021e-02,  

## BM25

In [14]:
def _tokenized(text: str):
    text = text.lower()

    # remove punctuation
    text = re.sub(r"[^\w\s]", "", text)

    text = text.split(" ")
    return text

def _cleaning(text: str):
    text = text.lower()

    # remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    
    return text

In [15]:
corpus = [_cleaning(d["text"]) for d in data]

tokenized_corpus = list(map(_tokenized, corpus))

print(corpus[0])
print(tokenized_corpus[0])

nama produk bumi playsuit baby bayi piyama pakaian tidur anak 1 tahun size  deskripsi playsuit ini terbuat dari bahan katun pilihan dan yang pasti terbaik untuk sang buah hatin kami pastikan playsuit ini memberikan kenyamanan pada sang buah hati sepanjang tahun di segala musim harga retail 490000 harga reseller diskon 15 416500 stockist diskon 30 343000 berat gram 1000 kategori jumper jumlah varian 7 varian warna army gambar httpsngorder1sgp1digitaloceanspacescom102804productsplaysuitbabybayipiyamapakaiantiduranak1tahun1700452110600png jumlah stok real 212 warna brick gambar httpsngorder1sgp1digitaloceanspacescom102804productsplaysuitbabybayipiyamapakaiantiduranak1tahun1700452192452png jumlah stok real 268 warna denim gambar httpsngorder1sgp1digitaloceanspacescom102804productsplaysuitbabybayipiyamapakaiantiduranak1tahun1700452170282jpg jumlah stok real 286 warna latte gambar httpsngorder1sgp1digitaloceanspacescom102804productsplaysuitbabybayipiyamapakaiantiduranak1tahun1700452165007png

In [16]:
bm25 = BM25Okapi(tokenized_corpus)

In [17]:
scores = bm25.get_scores(_tokenized(sample_query))
print(type(scores))
scores

<class 'numpy.ndarray'>


array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.26665121, 0.        , 2.11778449,
       0.        , 0.35977661, 4.75527612, 4.77033665, 0.        ,
       0.        , 5.33799041, 0.18675875, 0.20325721, 0.        ,
       3.88278291, 3.28099519, 3.28099519, 0.        , 0.        ,
       1.02173304, 0.21889797, 0.        , 0.        , 0.26125212,
       0.        , 3.88278291, 0.        , 0.        , 0.        ,
       2.11778449, 0.19760935, 0.        , 3.28099519, 0.        ,
       3.89965958, 4.77033665, 3.89965958, 2.11778449, 3.28099519,
       0.        , 0.        , 0.        , 0.23607307, 1.02173304,
       1.02173304, 0.        , 0.        ])

In [18]:
# get top 10
idx = scores.argsort()[-10:][::-1]
idx

array([16, 13, 41, 12, 42, 40, 31, 20, 21, 44])

In [19]:
candidates = [data[i] for i in idx]
candidates

[{'id': '16',
  'filename': 'Ara Casual Button Motif 6-12 Bulan.json',
  'text': "{'Nama Produk': 'Ara Casual Button Motif 6-12 Bulan', 'Size': '-', 'Deskripsi': 'casual button motif', 'Harga Retail': 74900.0, 'Harga Reseller (Diskon 15%)': 63665.0, 'Stockist (Diskon 30%)': 52430.0, 'Berat (gram)': 125.0, 'Kategori': 'Setelan Baju Anak Unisex', 'jumlah varian': 3, 'varian': [{'Warna': 'Mangga', 'Gambar': 'https://ngorder-1.sgp1.digitaloceanspaces.com/102804/products/ara-casual-button-motif-6-12-bulan-1712021789349.png', 'Jumlah Stok Real': 43}, {'Warna': 'Manggis', 'Gambar': 'https://ngorder-1.sgp1.digitaloceanspaces.com/102804/products/ara-casual-button-motif-6-12-bulan-1712021784556.png', 'Jumlah Stok Real': 37}, {'Warna': 'Rambutan', 'Gambar': 'https://ngorder-1.sgp1.digitaloceanspaces.com/102804/products/ara-casual-button-motif-6-12-bulan-1712021781664.png', 'Jumlah Stok Real': 39}]}"},
 {'id': '13',
  'filename': 'Ara Casual Button Motif 1 Tahun.json',
  'text': "{'Nama Produk': '

In [20]:
[scores[i] for i in idx]

[np.float64(5.337990406147155),
 np.float64(4.7703366519446595),
 np.float64(4.7703366519446595),
 np.float64(4.755276120045967),
 np.float64(3.8996595815084554),
 np.float64(3.8996595815084554),
 np.float64(3.882782911590387),
 np.float64(3.882782911590387),
 np.float64(3.2809951875956616),
 np.float64(3.2809951875956616)]

In [21]:
ranked_result = []
for i in idx:
    result = {
        "id": data[i]["id"],
        "text": data[i]["text"],
        "score": scores[i]
    }
    ranked_result.append(result)

ranked_result

[{'id': '16',
  'text': "{'Nama Produk': 'Ara Casual Button Motif 6-12 Bulan', 'Size': '-', 'Deskripsi': 'casual button motif', 'Harga Retail': 74900.0, 'Harga Reseller (Diskon 15%)': 63665.0, 'Stockist (Diskon 30%)': 52430.0, 'Berat (gram)': 125.0, 'Kategori': 'Setelan Baju Anak Unisex', 'jumlah varian': 3, 'varian': [{'Warna': 'Mangga', 'Gambar': 'https://ngorder-1.sgp1.digitaloceanspaces.com/102804/products/ara-casual-button-motif-6-12-bulan-1712021789349.png', 'Jumlah Stok Real': 43}, {'Warna': 'Manggis', 'Gambar': 'https://ngorder-1.sgp1.digitaloceanspaces.com/102804/products/ara-casual-button-motif-6-12-bulan-1712021784556.png', 'Jumlah Stok Real': 37}, {'Warna': 'Rambutan', 'Gambar': 'https://ngorder-1.sgp1.digitaloceanspaces.com/102804/products/ara-casual-button-motif-6-12-bulan-1712021781664.png', 'Jumlah Stok Real': 39}]}",
  'score': np.float64(5.337990406147155)},
 {'id': '13',
  'text': "{'Nama Produk': 'Ara Casual Button Motif 1 Tahun', 'Size': '-', 'Deskripsi': 'motif', 

# dense search

In [22]:
for i, d in enumerate(corpus):
    embedding = model.encode(d)
    data[i]["embedding"] = embedding

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
for d in data:
    print(d["id"])

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52


In [24]:
for d in data:
    db.insert_vectors(
        ids = [d["id"]],
        docs = [d["text"]],
        embeddings = [d["embedding"]]
    )

INFO:httpx:HTTP Request: GET http://localhost:8008/api/v2/pre-flight-checks "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8008/api/v2/tenants/default_tenant/databases/default_database/collections/bd60e9b7-caef-468e-a453-8d8d6590ea06/add "HTTP/1.1 201 Created"
INFO:httpx:HTTP Request: POST http://localhost:8008/api/v2/tenants/default_tenant/databases/default_database/collections/bd60e9b7-caef-468e-a453-8d8d6590ea06/add "HTTP/1.1 201 Created"
INFO:httpx:HTTP Request: POST http://localhost:8008/api/v2/tenants/default_tenant/databases/default_database/collections/bd60e9b7-caef-468e-a453-8d8d6590ea06/add "HTTP/1.1 201 Created"
INFO:httpx:HTTP Request: POST http://localhost:8008/api/v2/tenants/default_tenant/databases/default_database/collections/bd60e9b7-caef-468e-a453-8d8d6590ea06/add "HTTP/1.1 201 Created"
INFO:httpx:HTTP Request: POST http://localhost:8008/api/v2/tenants/default_tenant/databases/default_database/collections/bd60e9b7-caef-468e-a453-8d8d6590ea06/add "HTT

In [25]:
db.search_vectors(
    query = query,
    top_k = 5
)

INFO:httpx:HTTP Request: POST http://localhost:8008/api/v2/tenants/default_tenant/databases/default_database/collections/bd60e9b7-caef-468e-a453-8d8d6590ea06/query "HTTP/1.1 200 OK"


{'ids': [['16', '12', '22', '38', '44']],
 'distances': [[91.55177642455133,
   97.16965142850191,
   97.77966467718679,
   98.9137849437899,
   99.56415715073486]],
 'embeddings': None,
 'metadatas': [[None, None, None, None, None]],
 'documents': [["{'Nama Produk': 'Ara Casual Button Motif 6-12 Bulan', 'Size': '-', 'Deskripsi': 'casual button motif', 'Harga Retail': 74900.0, 'Harga Reseller (Diskon 15%)': 63665.0, 'Stockist (Diskon 30%)': 52430.0, 'Berat (gram)': 125.0, 'Kategori': 'Setelan Baju Anak Unisex', 'jumlah varian': 3, 'varian': [{'Warna': 'Mangga', 'Gambar': 'https://ngorder-1.sgp1.digitaloceanspaces.com/102804/products/ara-casual-button-motif-6-12-bulan-1712021789349.png', 'Jumlah Stok Real': 43}, {'Warna': 'Manggis', 'Gambar': 'https://ngorder-1.sgp1.digitaloceanspaces.com/102804/products/ara-casual-button-motif-6-12-bulan-1712021784556.png', 'Jumlah Stok Real': 37}, {'Warna': 'Rambutan', 'Gambar': 'https://ngorder-1.sgp1.digitaloceanspaces.com/102804/products/ara-casual